In [26]:
import re
import string

# ------------------------------------------------------------------
# 1) Regex to capture a leading paragraph number like "7." or "14."
# ------------------------------------------------------------------
paragraph_num_re = re.compile(r"^(\d+)\.\s*(.*)$")

# ------------------------------------------------------------------
# 2) Citation pattern: captures ECHR references
# ------------------------------------------------------------------
citation_pattern = re.compile(
    r"""
    # Optional leading bracket/parenthesis or "see ..." phrase
    (?P<leading_punct>
        [\(\[]?
        (?:see(?:\s+in\sparticular)?\s+)?
    )?

    # Case name, e.g. "Fedotova and Others v. Russia", "Denisov v. Ukraine"
    (?P<case_name>
        [A-Z][^\(\)\[\],]*      # start uppercase, then anything but parens/brackets/commas
        \s+v\.?\s+              # " v. ", " v ", or " v."
        [A-Z][^,\(\)\[]*
    )

    # Possibly bracketed content like "[GC]" or "[Committee]"
    (?P<parenthetical>
        (?:\s*\[[^\]]+\])?
    )?

    # Optional footnote digits, e.g. "[GC]7"
    (?P<footnote>
        (?:\d+)?
    )

    # Comma or space
    (?:,\s*)?

    # Application numbers: "nos. 40792/10 and 2 others", "no. 31250/02"
    (?P<application_numbers>
        (?:no\.|nos\.)[^,§]+
    )?

    # Optional comma
    (?:,\s*)?

    # Paragraph references: "§ 27", "§§ 68-73", "§§ 6873", etc.
    (?P<paragraphs>
        §§?.*?\d+
        (?:,\s?\d+)?        
        (?:[-–]\d+)?        
        (?:\s*(?:and|,)\s*§§?\d+(?:[-–]\d+)?)* 
    )?

    # Optional comma
    (?:,\s*)?

    # Date: e.g. "4 March 2008" or "17 January 2023"
    (?P<date>
        (?:\d{1,2}\s+(?:January|February|March|April|May|June|July|
                      August|September|October|November|December)\s+\d{4})
    )?

    # Optional trailing punctuation, e.g. ")" or "]"
    (?P<trailing_punct>
        [\)\]]?
    )?
    """,
    re.VERBOSE
)


# ------------------------------------------------------------------
# 3a) Helper function: fix "§§ 6873" -> "§§ 68-73"
# ------------------------------------------------------------------
def fix_paragraphs(paragraphs_string: str) -> str:
    if not paragraphs_string:
        return ""
    pattern_missing_dash = re.compile(r"(§§?\s*)(\d{2})(\d{2})\b")
    def insert_dash(m):
        return f"{m.group(1)}{m.group(2)}-{m.group(3)}"
    paragraphs_fixed = pattern_missing_dash.sub(insert_dash, paragraphs_string)
    return paragraphs_fixed.strip()


# ------------------------------------------------------------------
# 3b) Handle editorial short form e.g. "115-31" => "115-131"
# ------------------------------------------------------------------
def expand_editorial_shortening(start_par: int, end_par: int) -> int:
    if end_par >= start_par:
        return end_par
    # If end_par < start_par, we suspect missing digits
    s_start = str(start_par)
    s_end = str(end_par)
    diff = len(s_start) - len(s_end)
    if diff <= 0:
        return end_par
    prefix = s_start[:-len(s_end)]  # e.g. '115'[:-2] => '1'
    new_str = prefix + s_end        # => '131'
    try:
        new_end = int(new_str)
        return new_end
    except ValueError:
        return end_par


# ------------------------------------------------------------------
# 3c) Parse paragraph references into (start_par, end_par)
# ------------------------------------------------------------------
def parse_paragraph_range(paragraphs_string: str) -> (int, int):
    if not paragraphs_string:
        return (None, None)
    fixed_string = fix_paragraphs(paragraphs_string)

    # e.g. "§§ 115-31, §§ 200-08" => split on commas/and, parse first chunk
    references = re.split(r",\s*|\sand\s+", fixed_string)
    if not references:
        return (None, None)

    first_chunk = references[0].strip()  # e.g. "§§ 115-31"

    # remove '§', '§§', spaces => keep digits/dash => e.g. "115-31"
    step1 = re.sub(r'§+', '', first_chunk)
    chunk_clean = re.sub(r'[^\d-]', '', step1)

    if '-' in chunk_clean:
        parts = chunk_clean.split('-', 1)
        if len(parts) == 2:
            try:
                start_par = int(parts[0])
            except ValueError:
                return (None, None)
            try:
                end_par = int(parts[1])
            except ValueError:
                return (start_par, None)

            if end_par < start_par:
                end_par = expand_editorial_shortening(start_par, end_par)
            return (start_par, end_par)
        else:
            return (None, None)
    else:
        # single number
        try:
            val = int(chunk_clean)
            return (val, val)
        except ValueError:
            return (None, None)


# ------------------------------------------------------------------
# 4) The KEY new piece: split on "<date> and " => separate citations
# ------------------------------------------------------------------
date_and_pattern = re.compile(
    r'(\d{1,2}\s+(?:January|February|March|April|May|June|July|August|'
    r'September|October|November|December)\s+\d{4})(?:,\s*)?\s+and\s+'
)

def split_on_date_and(text: str) -> list:
    """
    Whenever we see something like '25 September 2018 and <Case>...',
    we insert a separator '|||' so we can parse them as separate chunks.
    """
    # Sub: "25 September 2018 and " => "25 September 2018|||"
    replaced = date_and_pattern.sub(r'\1|||', text)
    # Now split on '|||'
    return replaced.split('|||')


# ------------------------------------------------------------------
# 5) Function to parse ECHR citations from a single text chunk
# ------------------------------------------------------------------
def parse_echr_citations_in_text(text: str):
    """
    Run the main citation_pattern on the given text (a single chunk or paragraph).
    Return a list of dictionaries for each match.
    """
    citations = []
    for match in citation_pattern.finditer(text):
        d = match.groupdict()
        full_citation = match.group(0)

        case_name = d.get('case_name') or ""
        parenthetical = d.get('parenthetical') or None
        footnote = d.get('footnote') or None
        application_numbers = d.get('application_numbers') or None
        paragraphs_str = d.get('paragraphs') or ""
        date_ = d.get('date') or None

        # separate out [Committee] if present
        committee = None
        if parenthetical and '[Committee]' in parenthetical:
            committee = 'Committee'
            parenthetical = parenthetical.replace('[Committee]', '').strip() or None

        paragraphs_fixed = fix_paragraphs(paragraphs_str)
        (start_par_ref, end_par_ref) = parse_paragraph_range(paragraphs_str)

        start_idx = match.start()
        end_idx = match.end()

        citation_info = {
            'full_citation': full_citation,
            'case_name': case_name.strip(),
            'parenthetical': parenthetical,
            'committee': committee,
            'footnote': footnote,
            'application_numbers': application_numbers.strip() if application_numbers else None,
            'paragraphs': paragraphs_fixed,
            'start_paragraph_ref': start_par_ref,
            'end_paragraph_ref': end_par_ref,
            'date': date_,
            'start': start_idx,
            'end': end_idx
        }
        citations.append(citation_info)

    return citations


# ------------------------------------------------------------------
# 6) Main entry: extract from a list of paragraphs
# ------------------------------------------------------------------
def extract_echr_citations(law_paragraphs):
    """
    law_paragraphs: list of paragraph strings (some may start with "7. " or "122. ").
    We'll:
      1) detect paragraph number
      2) split the text on the date+and pattern
      3) parse each chunk with parse_echr_citations_in_text
      4) combine results (optionally deduplicate if needed)
    """
    all_citations = []

    for para_text in law_paragraphs:
        paragraph_number = None
        match_parnum = paragraph_num_re.match(para_text)
        if match_parnum:
            paragraph_number = match_parnum.group(1)
            text_body = match_parnum.group(2)
        else:
            text_body = para_text

        # Split the paragraph on <date> + "and" => each chunk might contain 0 or 1 reference
        chunks = split_on_date_and(text_body)

        for c in chunks:
            c = c.strip()
            # parse the chunk with citation_pattern
            chunk_citations = parse_echr_citations_in_text(c)
            # store results
            for cit in chunk_citations:
                # Add the paragraph number to the final result
                all_citations.append({
                    'paragraph': paragraph_number,
                    'citation': cit
                })
    
    return all_citations


In [27]:
import pandas as pd
import json

def load_json(file_path: str):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

              

data_path = '/Users/ahmed/Desktop/msc-24/ECHR_v2/echr_processed/'
df1_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_1.csv'
df4_path = '/Users/ahmed/Desktop/msc-24/TND/kc_classification_data/pre_cutoff_data/df_4.csv'
df1 = pd.read_csv(df1_path)['file_path'].to_list()
df4 = pd.read_csv(df4_path)['file_path'].to_list()


In [42]:
case_path = data_path + '001-218424.json'
case = load_json(case_path)
text_list = case['law']

print(case['itemid'])

results = extract_echr_citations(text_list)

for item in results:
    print(item)

001-218424
{'paragraph': '95', 'citation': {'full_citation': '(see V.M. and Others v. Belgium ', 'case_name': 'V.M. and Others v. Belgium', 'parenthetical': None, 'committee': None, 'footnote': None, 'application_numbers': None, 'paragraphs': '', 'start_paragraph_ref': None, 'end_paragraph_ref': None, 'date': None, 'start': 489, 'end': 521}}
{'paragraph': '95', 'citation': {'full_citation': 'November 2016; Sharifi and Others v. Italy and Greece, no. 16643/09, § 124, 21', 'case_name': 'November 2016; Sharifi and Others v. Italy and Greece', 'parenthetical': None, 'committee': None, 'footnote': None, 'application_numbers': 'no. 16643/09', 'paragraphs': '§ 124, 21', 'start_paragraph_ref': 124, 'end_paragraph_ref': 124, 'date': None, 'start': 565, 'end': 643}}
{'paragraph': '95', 'citation': {'full_citation': 'Ali v. Switzerland, 5 August 1998', 'case_name': 'Ali v. Switzerland', 'parenthetical': None, 'committee': None, 'footnote': None, 'application_numbers': None, 'paragraphs': '', 'sta

In [30]:
case_path = data_path + df4[11]
case = load_json(case_path)
text_list = case['facts']

print(case['itemid'])

results = extract_echr_citations(text_list)

for item in results:
    print(item)

001-229327
